# Assignment 5 Solution

Backtest example strategy for the Swiss equity market.

In [ ]:
import os, sys
project_root = os.path.abspath('..')
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, 'src'))

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from helper_functions import load_data_spi
from estimation.covariance import Covariance
from estimation.expected_return import ExpectedReturn
from optimization.optimization import MeanVariance
from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest_item_builder_classes import SelectionItemBuilder, OptimizationItemBuilder
from backtesting.backtest_item_builder_functions import (
    bibfn_selection_min_volume,
    bibfn_return_series,
    bibfn_budget_constraint,
    bibfn_box_constraints,
    bibfn_size_dependent_upper_bounds,
    bibfn_turnover_constraint,
)


## Load data

In [ ]:

# Load market and factor data
market_data = pd.read_parquet('../data/market_data.parquet')
jkp_data = pd.read_parquet('../data/jkp_data.parquet')

# Load benchmark series (Swiss Performance Index)
bm_series = load_data_spi(path='../data/')

# Wrap data in BacktestData object
bt_data = BacktestData()
bt_data.market_data = market_data
bt_data.jkp_data = jkp_data
bt_data.bm_series = bm_series


## Set up backtest service

In [ ]:

# Instantiate expected return and covariance models
expected_return = ExpectedReturn(method='geometric')
covariance = Covariance(method='pearson')

# Rebalancing dates (quarterly)
dates = market_data.index.get_level_values('date').unique().sort_values()
rebdates = dates[dates > '2010-01-01'][::63].strftime('%Y-%m-%d').tolist()

# Selection builders: remove illiquid stocks
selection_item_builders = {
    'min_volume': SelectionItemBuilder(
        bibfn=bibfn_selection_min_volume,
        width=252,
        min_volume=500000,
    )
}

# Optimization item builders
optimization_item_builders = {
    'return_series': OptimizationItemBuilder(bibfn=bibfn_return_series, width=252),
    'budget_constraint': OptimizationItemBuilder(bibfn=bibfn_budget_constraint, budget=1),
    'box_constraints': OptimizationItemBuilder(bibfn=bibfn_box_constraints, upper=0.1),
    'size_bounds': OptimizationItemBuilder(bibfn=bibfn_size_dependent_upper_bounds),
    'turnover_constraint': OptimizationItemBuilder(bibfn=bibfn_turnover_constraint, turnover_limit=0.25),
}

# Create backtest service
test_service = BacktestService(
    data=bt_data,
    optimization_item_builders=optimization_item_builders,
    selection_item_builders=selection_item_builders,
    rebdates=rebdates,
    optimization=MeanVariance(
        expected_return=expected_return,
        covariance=covariance,
        risk_aversion=1,
        solver_name='cvxopt',
    ),
)


## Run backtest

In [ ]:

from backtesting.backtest import Backtest

bt = Backtest()
bt.run(test_service)


## Simulate portfolio

In [ ]:

# Combine strategy returns with benchmark
returns = bt.strategy.simulate(
    return_series=test_service.data.get_return_series(),
    fc=0.01,
    vc=0.002,
)

sim = pd.concat({'bm': bm_series, 'strategy': returns}, axis=1).dropna()

(np.log1p(sim)).cumsum().plot(title='Cumulative Performance', figsize=(10,6))
plt.show()


## Performance statistics

In [ ]:

def perf_stats(x):
    ann_ret = np.exp(np.log1p(x).mean()*252)-1
    ann_vol = x.std()*np.sqrt(252)
    sharpe = ann_ret/ann_vol
    cum_ret = (1+x).prod()-1
    running = (1+x).cumprod()
    peak = running.cummax()
    drawdown = running/peak -1
    max_dd = drawdown.min()
    return pd.Series({'annual_return':ann_ret,'annual_volatility':ann_vol,'sharpe':sharpe,'cumulative_return':cum_ret,'max_drawdown':max_dd})

stats = sim.apply(perf_stats)
print(stats)
